# aug27 — memorization control on Kaggle T4 x2 (the C11-style offline train-set scoring run)

**Settings → Accelerator → GPU T4 x2, Internet → On.** Secrets: `HF_TOKEN` (required). Nothing else.
No training happens in this notebook — it scores checkpoints that already exist.

**The question this run decides** (paper §10, register issue D): training reward hits its
ceiling of 1.6 by step 125 while the final policy solves 0/162 test puzzles. If the SFT stage
(3 epochs over the 807 train puzzles) memorized the answer keys, "training reward 1.6,
held-out 0.125" is an ordinary train/test gap and the construct-validity thesis weakens.
Neither SFT nor the final policy has ever been scored offline on training puzzles.

**Design:** ONE vLLM session ("**aug27 memC session**" — every number carries this label).
Four arms — base, SFT, GRPO step 50, GRPO final — greedy (T=0.0), scored with the training
reward on **(a)** the full 807-puzzle train split and **(b)** the 162-puzzle test split
re-served in the same session, so every train-vs-test contrast is within-session.
The prespecified 162-puzzle train slice (`c11_train_sample.json`, frozen 2026-08-21) is
reported as a size-matched secondary readout.

**How to read the outcome** (written before running):

| Result | Reading |
|---|---|
| `grpo-final` train ≈ ceiling (solves train greedily) | memorization real: the 1.6 reflects genuine train-answer knowledge; the reversal is a train/test gap and §10's open flag resolves against us — the paper must say so |
| `grpo-final` train ≈ its test score (near floor) | the sampling-time ceiling does not correspond to greedy train knowledge; not simple memorization; §10's flag resolves in the construct-validity story's favor |
| `sft` train ≫ `sft` test | memorization begins at the SFT stage (3 epochs); report the gap |
| `sft` train ≈ `sft` test | SFT did not memorize; the train signal is capability, not lookup |
| `base` train ≉ `base` test | sanity failure — base saw no training data; if these differ beyond CI overlap, distrust the whole session |

Time budget ~7–8 h against Kaggle's 12 h cap (≈3,900 greedy generations at the aug21
session's throughput). The train evals run FIRST so a session death still yields the
decisive readout. If the session dies: re-import and Run All — everything recomputes.


In [ ]:
# Cell 1 — setup. No `huggingface-cli login` line (it can hang on an interactive
# update prompt); the HF_TOKEN env var is sufficient everywhere it is needed.
import os, glob, json, subprocess, time
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
HF_USER = 'jacksonlukas'

!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
# The prespecified slice + aug21 files live on analysis/aug21 if not yet merged:
if not os.path.exists('results-analysis/aug21/c11_train_sample.json'):
    !git checkout analysis/aug21
assert os.path.exists('results-analysis/aug21/c11_train_sample.json'), \
    'c11_train_sample.json missing on every branch tried -- push it first'
!pip install -q -e . openai vllm peft accelerate
!pip show vllm | grep -E '^(Name|Version)'
# Editable-install .pth files are invisible to the already-running kernel:
# verify the import in a SUBPROCESS, exactly as the eval commands will run.
r = subprocess.run(['python', '-c', 'import connections_rl; print("import OK")'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr); assert r.returncode == 0, 'connections_rl not importable in subprocess'

!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!make data
cands = sorted(glob.glob('data/splits/*train*.json'))
assert len(cands) == 1, f'expected exactly one train split file, found {cands}'
TRAIN_SPLIT = cands[0]
r = subprocess.run(['python', '-c', (
    'from connections_rl.data.loader import load_puzzles;'
    f'print(len(load_puzzles("{TRAIN_SPLIT}")))')], capture_output=True, text=True)
n_train = int(r.stdout.strip()); print('train split:', TRAIN_SPLIT, '->', n_train, 'puzzles')
assert n_train == 807, f'train split has {n_train} puzzles, expected 807 -- wrong DB or split code'
r = subprocess.run(['python', '-c', (
    'from connections_rl.data.loader import load_puzzles;'
    'print(len(load_puzzles("data/splits/puzzles_test.json")))')], capture_output=True, text=True)
assert int(r.stdout.strip()) == 162, 'test split is not 162 puzzles'
print('splits verified: 807 train / 162 test')


In [ ]:
# Cell 2 -- adapters from the Hub, then ONE vLLM session for everything.
import subprocess, time, urllib.request, os
from huggingface_hub import snapshot_download
snapshot_download(f'{HF_USER}/connections-rl-sft-7b', local_dir='adapters/sft-7b',
                  token=os.environ['HF_TOKEN'])
snapshot_download(f'{HF_USER}/connections-rl-grpo-7b-ckpt', local_dir='adapters/grpo-7b-ckpt',
                  allow_patterns='checkpoint-50/*', token=os.environ['HF_TOKEN'])
snapshot_download(f'{HF_USER}/connections-rl-grpo-7b', local_dir='adapters/grpo-7b',
                  token=os.environ['HF_TOKEN'])
assert os.path.exists('adapters/sft-7b/adapter_config.json'), 'SFT adapter incomplete'
assert os.path.isdir('adapters/grpo-7b-ckpt/checkpoint-50'), 'ckpt-50 missing from ckpt repo'

mods = ['connections-rl-sft-7b=adapters/sft-7b',
        'connections-rl-grpo-7b-ckpt50=adapters/grpo-7b-ckpt/checkpoint-50',
        'connections-rl-grpo-7b=adapters/grpo-7b']
proc = subprocess.Popen(
    'vllm serve Qwen/Qwen2.5-7B-Instruct --dtype half --tensor-parallel-size 2 '
    '--enable-lora --enforce-eager --max-lora-rank 16 --max-model-len 2048 '
    '--gpu-memory-utilization 0.85 --lora-modules ' + ' '.join(mods),
    shell=True, stdout=open('/kaggle/working/vllm.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(150):
    try:
        urllib.request.urlopen('http://localhost:8000/health'); print('vLLM ready'); break
    except Exception:
        time.sleep(10)
else:
    raise RuntimeError('vLLM failed to come up -- see /kaggle/working/vllm.log')
!mkdir -p results-analysis/aug27
!pip show vllm | grep -E '^(Name|Version)' | tee results-analysis/aug27/memC-session-versions.txt


In [ ]:
# Cell 3 -- the evals: train FIRST (the decisive readout), then test, same session.
import os, subprocess, time
os.makedirs('results-analysis/aug27', exist_ok=True)
ARMS = ['base', 'sft', 'grpo-ckpt50', 'grpo-final']
MODELS = {'base': 'Qwen/Qwen2.5-7B-Instruct',
          'sft': 'connections-rl-sft-7b',
          'grpo-ckpt50': 'connections-rl-grpo-7b-ckpt50',
          'grpo-final': 'connections-rl-grpo-7b'}
def write_cfg(path, puzzles, out_dir):
    lines = [f'puzzles: {puzzles}', f'out_dir: {out_dir}',
             'n_resamples: 1000', 'seed: 0', 'capture_generations: true', '', 'arms:']
    for a in ARMS:
        lines += [f'  - name: {a}', f'    model: {MODELS[a]}', '    temperature: 0.0']
    open(path, 'w').write('\n'.join(lines) + '\n')
write_cfg('results-analysis/aug27/memC_train.yaml', TRAIN_SPLIT,
          'results-analysis/aug27/memC-session-train')
write_cfg('results-analysis/aug27/memC_test.yaml', 'data/splits/puzzles_test.json',
          'results-analysis/aug27/memC-session-test')
for cfg in ('memC_train', 'memC_test'):
    t0 = time.time()
    r = subprocess.run(['python', '-m', 'connections_rl.eval.run',
                        '--config', f'results-analysis/aug27/{cfg}.yaml'])
    print(f'{cfg}: exit {r.returncode} in {(time.time()-t0)/60:.0f} min')
    assert r.returncode == 0, f'{cfg} failed'


In [ ]:
# Cell 4 -- analysis. Every number below is from THIS session (aug27 memC session).
import json, random
SLICE_IDS = set(json.load(open('results-analysis/aug21/c11_train_sample.json'))['sample_ids'])
assert len(SLICE_IDS) == 162

def load_arm(split, arm):
    d = f'results-analysis/aug27/memC-session-{split}/{arm}'
    m = json.load(open(f'{d}/metrics.json'))
    recs = [json.loads(l) for l in open(f'{d}/records.jsonl')]
    return m, recs

def boot_ci(vals, n_resamples=1000, seed=0):
    rng = random.Random(seed); n = len(vals)
    means = sorted(sum(rng.choices(vals, k=n)) / n for _ in range(n_resamples))
    return means[int(0.025 * n_resamples)], means[int(0.975 * n_resamples)]

summary = {'session': 'aug27 memC session', 'decoding': 'greedy T=0.0',
           'splits': {'train': 807, 'test': 162, 'train_slice': 162}, 'arms': {}}
hdr = f"{'arm':<12} {'split':<12} {'groups(0-4)':<22} {'slots':<10} {'invalid':<8} {'reward':<20} solves"
print(hdr); print('-' * len(hdr))
for arm in ARMS:
    summary['arms'][arm] = {}
    for split in ('train', 'test'):
        m, recs = load_arm(split, arm)
        s = m['summary']['OVERALL']; n = m['n']
        slots = round(s['groups_correct'][0] * n)
        solves = sum(r['solved'] for r in recs)
        row = {'n': n, 'groups_mean_ci': s['groups_correct'], 'slots': slots,
               'slots_of': 4 * n, 'invalid_rate_ci': s['invalid_rate'],
               'reward_mean_ci': s['reward'], 'solves': solves}
        summary['arms'][arm][split] = row
        print(f"{arm:<12} {split:<12} {s['groups_correct'][0]:.4f} [{s['groups_correct'][1]:.3f},{s['groups_correct'][2]:.3f}]  "
              f"{slots}/{4*n:<6} {s['invalid_rate'][0]:<8.3f} "
              f"{s['reward'][0]:.4f} [{s['reward'][1]:.3f},{s['reward'][2]:.3f}]  {solves}/{n}")
        if split == 'train':
            sl = [r for r in recs if r['puzzle_id'] in SLICE_IDS]
            assert len(sl) == 162, f'{arm}: slice matched {len(sl)} of 162'
            g = [float(r['groups_correct']) for r in sl]
            rw = [r['reward'] for r in sl]
            glo, ghi = boot_ci(g); mean_g = sum(g) / len(g)
            row_s = {'n': 162, 'groups_mean_ci': [mean_g, glo, ghi],
                     'slots': round(mean_g * 162), 'slots_of': 648,
                     'reward_mean': sum(rw) / len(rw),
                     'solves': sum(r['solved'] for r in sl)}
            summary['arms'][arm]['train_slice'] = row_s
            print(f"{arm:<12} {'train-slice':<12} {mean_g:.4f} [{glo:.3f},{ghi:.3f}]  "
                  f"{row_s['slots']}/648    {'':<8} {row_s['reward_mean']:.4f}")

print()
print('=== DECISIVE READOUTS (all within the aug27 memC session) ===')
ftr = summary['arms']['grpo-final']['train']; fte = summary['arms']['grpo-final']['test']
str_ = summary['arms']['sft']['train'];      ste = summary['arms']['sft']['test']
btr = summary['arms']['base']['train'];      bte = summary['arms']['base']['test']
print(f"grpo-final train reward {ftr['reward_mean_ci'][0]:.4f} vs ceiling 1.6; "
      f"train groups {ftr['groups_mean_ci'][0]:.4f} vs test {fte['groups_mean_ci'][0]:.4f}")
print(f"sft        train groups {str_['groups_mean_ci'][0]:.4f} "
      f"[{str_['groups_mean_ci'][1]:.3f},{str_['groups_mean_ci'][2]:.3f}] vs "
      f"test {ste['groups_mean_ci'][0]:.4f} [{ste['groups_mean_ci'][1]:.3f},{ste['groups_mean_ci'][2]:.3f}]")
print(f"base       train groups {btr['groups_mean_ci'][0]:.4f} vs test {bte['groups_mean_ci'][0]:.4f}"
      '  <- sanity: these should be close (CI overlap)')
print()
print('Cross-session reference (do NOT mix into the rows above): c11 base-trainslice,')
print('aug21 session, same 162-id slice, groups 0.1481 [0.086, 0.222]; published test')
print('anchors (Task D session): base 26/648, sft 56/648 & 52/648 by session, ckpt50 77/648, final 4/648.')
json.dump(summary, open('results-analysis/aug27/memorization_control.json', 'w'), indent=1)
print()
print('wrote results-analysis/aug27/memorization_control.json')


In [ ]:
# Cell 5 -- persist. Zip for download; Hub upload attempted but never required.
!zip -qr /kaggle/working/aug27-memC-outputs.zip results-analysis/aug27
print('zip ready: /kaggle/working/aug27-memC-outputs.zip  (download from the Output tab)')
try:
    from huggingface_hub import HfApi
    HfApi(token=os.environ['HF_TOKEN']).upload_folder(
        folder_path='results-analysis/aug27',
        repo_id=f'{HF_USER}/connections-rl-artifacts', repo_type='dataset',
        path_in_repo='aug27')
    print('Hub upload OK -> connections-rl-artifacts/aug27')
except Exception as e:
    print('Hub upload skipped/failed (fine -- use the zip):', e)


## What to bring back

`aug27-memC-outputs.zip` (or the Hub `aug27/` folder). The one file the paper edit needs is
`memorization_control.json`; the per-arm `records.jsonl` + `generations.jsonl` are the audit
trail. Rules that apply when writing it up: every number names this session (**aug27 memC
session**); train-vs-test contrasts here are within-session but **unpaired** (different puzzle
sets — no paired bootstrap across splits); the slice rows exist for size-matched comparison
against c11's base-trainslice, which is a different session and is never averaged with these.
